# Week 3 Exercise - Synthetic Technical Interview Dataset Generator

This notebook generates synthetic technical interview Q&A datasets for training and evaluation purposes.

## Features
- Configurable schema for different interview types (Python, ML, System Design)
- Rule-based generation for fast, deterministic data
- LLM-powered generation for realistic, diverse responses
- Data validation and quality checks
- Export to CSV/JSON formats

## Use Cases
- Training data for interview prep chatbots
- Evaluation datasets for technical assessments
- Fine-tuning data for domain-specific models

In [ ]:
# Imports

import os
import re
import json
import time
import uuid
import random
from datetime import datetime
from typing import List, Dict, Any, Optional
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully")

In [ ]:
# Environment Setup

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    USE_LLM = True
else:
    print("OpenAI API Key not set - will use rule-based generation only")
    USE_LLM = False

# Setup client (works with OpenRouter or direct OpenAI)
if USE_LLM:
    if openai_api_key.startswith('sk-or-'):
        # OpenRouter
        client = OpenAI(
            api_key=openai_api_key,
            base_url="https://openrouter.ai/api/v1"
        )
        MODEL = "gpt-4.1-mini"
    else:
        # Direct OpenAI
        client = OpenAI()
        MODEL = "gpt-4o-mini"
    print(f"Using model: {MODEL}")

In [ ]:
# Configuration Schema

CONFIG = {
    "dataset_name": "technical_interview_qa",
    "total_rows": 100,
    "batch_size": 10,
    
    "categories": [
        {"name": "python_fundamentals", "weight": 0.25},
        {"name": "data_structures", "weight": 0.20},
        {"name": "algorithms", "weight": 0.15},
        {"name": "machine_learning", "weight": 0.20},
        {"name": "system_design", "weight": 0.10},
        {"name": "llm_engineering", "weight": 0.10}
    ],
    
    "difficulty_levels": [
        {"level": "easy", "weight": 0.30},
        {"level": "medium", "weight": 0.50},
        {"level": "hard", "weight": 0.20}
    ],
    
    "fields": [
        {"name": "id", "type": "uuid"},
        {"name": "category", "type": "enum", "values": ["python_fundamentals", "data_structures", "algorithms", "machine_learning", "system_design", "llm_engineering"]},
        {"name": "difficulty", "type": "enum", "values": ["easy", "medium", "hard"]},
        {"name": "question", "type": "text"},
        {"name": "answer", "type": "text"},
        {"name": "code_example", "type": "text", "optional": True},
        {"name": "follow_up_questions", "type": "list"},
        {"name": "tags", "type": "list"},
        {"name": "estimated_time_minutes", "type": "int", "min": 5, "max": 45}
    ]
}

print(f"Configuration loaded:")
print(f"  - Dataset: {CONFIG['dataset_name']}")
print(f"  - Target rows: {CONFIG['total_rows']}")
print(f"  - Categories: {len(CONFIG['categories'])}")
print(f"  - Fields: {len(CONFIG['fields'])}")

In [ ]:
# Question Templates for Rule-Based Generation

QUESTION_TEMPLATES = {
    "python_fundamentals": [
        "Explain the difference between {concept1} and {concept2} in Python.",
        "What is {concept} in Python and when would you use it?",
        "How does Python handle {topic}?",
        "What are the best practices for {topic} in Python?",
        "Explain how {feature} works under the hood in Python."
    ],
    "data_structures": [
        "Explain the time complexity of {operation} on a {data_structure}.",
        "When would you use a {data_structure} instead of a {alt_structure}?",
        "How would you implement a {data_structure} from scratch?",
        "What are the trade-offs between {structure1} and {structure2}?"
    ],
    "algorithms": [
        "Explain how {algorithm} works and its time complexity.",
        "When would you use {algorithm} over {alt_algorithm}?",
        "How would you optimize {algorithm} for {scenario}?",
        "Walk through the steps of {algorithm} with an example."
    ],
    "machine_learning": [
        "Explain the difference between {model1} and {model2}.",
        "How would you handle {problem} in a machine learning pipeline?",
        "What metrics would you use to evaluate {task_type}?",
        "Explain {concept} and its importance in ML."
    ],
    "system_design": [
        "How would you design a {system_type}?",
        "What are the key considerations for scaling {component}?",
        "How would you ensure {quality_attribute} in {system}?",
        "Describe the trade-offs in designing {feature}."
    ],
    "llm_engineering": [
        "Explain how {concept} works in LLMs.",
        "What are best practices for {task} with LLMs?",
        "How would you implement {feature} using LLM APIs?",
        "Compare {approach1} vs {approach2} for LLM applications."
    ]
}

CONCEPT_POOLS = {
    "python_fundamentals": {
        "concept1": ["list", "tuple", "set", "dict", "generator", "iterator"],
        "concept2": ["tuple", "set", "dict", "list", "comprehension", "lambda"],
        "concept": ["decorators", "context managers", "generators", "metaclasses", "descriptors", "GIL"],
        "topic": ["memory management", "exception handling", "inheritance", "concurrency", "type hints"],
        "feature": ["garbage collection", "import system", "method resolution order", "slots"]
    },
    "data_structures": {
        "operation": ["insertion", "deletion", "search", "traversal"],
        "data_structure": ["hash table", "binary tree", "linked list", "heap", "graph", "trie"],
        "alt_structure": ["array", "balanced tree", "skip list", "B-tree"],
        "structure1": ["array", "linked list", "hash map", "tree"],
        "structure2": ["linked list", "array", "tree", "hash map"]
    },
    "algorithms": {
        "algorithm": ["quicksort", "mergesort", "binary search", "BFS", "DFS", "Dijkstra's", "dynamic programming"],
        "alt_algorithm": ["heapsort", "linear search", "A*", "greedy approach"],
        "scenario": ["large datasets", "memory constraints", "real-time systems", "distributed systems"]
    },
    "machine_learning": {
        "model1": ["Random Forest", "XGBoost", "Neural Network", "SVM", "Logistic Regression"],
        "model2": ["Decision Tree", "LightGBM", "Linear Regression", "KNN", "Naive Bayes"],
        "problem": ["class imbalance", "overfitting", "missing data", "feature selection", "hyperparameter tuning"],
        "task_type": ["classification", "regression", "clustering", "ranking"],
        "concept": ["bias-variance tradeoff", "regularization", "cross-validation", "feature engineering"]
    },
    "system_design": {
        "system_type": ["URL shortener", "chat application", "rate limiter", "cache system", "search engine"],
        "component": ["database", "API gateway", "message queue", "load balancer"],
        "quality_attribute": ["high availability", "consistency", "fault tolerance", "low latency"],
        "system": ["distributed system", "microservices", "event-driven architecture"],
        "feature": ["real-time notifications", "data replication", "authentication system"]
    },
    "llm_engineering": {
        "concept": ["attention mechanism", "tokenization", "embeddings", "prompt engineering", "fine-tuning"],
        "task": ["prompt optimization", "context management", "output parsing", "error handling"],
        "feature": ["streaming responses", "function calling", "RAG pipeline", "agent workflows"],
        "approach1": ["fine-tuning", "few-shot prompting", "RAG"],
        "approach2": ["prompt engineering", "zero-shot", "direct API calls"]
    }
}

print(f"Templates loaded: {sum(len(v) for v in QUESTION_TEMPLATES.values())} question templates")

In [ ]:
# Helper Functions

def weighted_choice(items: List[Dict], weight_key: str = "weight") -> Dict:
    """Select item based on weights."""
    weights = [item[weight_key] for item in items]
    return random.choices(items, weights=weights, k=1)[0]


def fill_template(template: str, concept_pool: Dict[str, List[str]]) -> str:
    """Fill template placeholders with random concepts."""
    result = template
    for key, values in concept_pool.items():
        placeholder = "{" + key + "}"
        if placeholder in result:
            result = result.replace(placeholder, random.choice(values), 1)
    return result


def generate_tags(category: str, difficulty: str) -> List[str]:
    """Generate relevant tags for a question."""
    base_tags = [category.replace("_", " "), difficulty]
    
    extra_tags_pool = {
        "python_fundamentals": ["core python", "syntax", "best practices", "pythonic"],
        "data_structures": ["complexity", "implementation", "optimization"],
        "algorithms": ["problem solving", "optimization", "complexity analysis"],
        "machine_learning": ["modeling", "evaluation", "preprocessing"],
        "system_design": ["scalability", "architecture", "distributed systems"],
        "llm_engineering": ["AI", "NLP", "transformers", "API integration"]
    }
    
    extra = random.sample(extra_tags_pool.get(category, []), k=min(2, len(extra_tags_pool.get(category, []))))
    return base_tags + extra


def estimate_time(difficulty: str, category: str) -> int:
    """Estimate time to answer based on difficulty."""
    base_times = {"easy": (5, 15), "medium": (15, 30), "hard": (25, 45)}
    min_t, max_t = base_times.get(difficulty, (10, 20))
    
    # System design and LLM questions tend to take longer
    if category in ["system_design", "llm_engineering"]:
        min_t += 5
        max_t += 10
    
    return random.randint(min_t, max_t)


print("Helper functions defined")

In [ ]:
# Rule-Based Generator

def generate_rule_based_row(config: Dict) -> Dict[str, Any]:
    """Generate a single row using rule-based logic."""
    # Select category and difficulty
    category_info = weighted_choice(config["categories"])
    category = category_info["name"]
    
    difficulty_info = weighted_choice(config["difficulty_levels"], "weight")
    difficulty = difficulty_info["level"]
    
    # Generate question from template
    template = random.choice(QUESTION_TEMPLATES[category])
    question = fill_template(template, CONCEPT_POOLS[category])
    
    # Generate placeholder answer (rule-based can only generate structure)
    answer = f"[Placeholder answer for: {question[:50]}...]"
    
    # Generate follow-up questions
    follow_ups = [
        f"Can you provide a code example?",
        f"What are common mistakes to avoid?",
        f"How does this apply in production?"
    ]
    
    return {
        "id": str(uuid.uuid4()),
        "category": category,
        "difficulty": difficulty,
        "question": question,
        "answer": answer,
        "code_example": None,
        "follow_up_questions": random.sample(follow_ups, k=2),
        "tags": generate_tags(category, difficulty),
        "estimated_time_minutes": estimate_time(difficulty, category)
    }


def generate_rule_based_dataset(config: Dict, n_rows: int) -> pd.DataFrame:
    """Generate dataset using rule-based approach."""
    rows = [generate_rule_based_row(config) for _ in range(n_rows)]
    return pd.DataFrame(rows)


# Test rule-based generation
print("Testing rule-based generation...")
test_df = generate_rule_based_dataset(CONFIG, 5)
print(f"Generated {len(test_df)} rows")
print(test_df[["category", "difficulty", "question"]].head())

In [ ]:
# LLM-Powered Generator

def create_generation_prompt(category: str, difficulty: str, n_items: int = 5) -> str:
    """Create prompt for LLM generation."""
    
    difficulty_guidance = {
        "easy": "suitable for junior developers, focusing on fundamentals",
        "medium": "suitable for mid-level developers, requiring practical experience",
        "hard": "suitable for senior developers, requiring deep expertise"
    }
    
    category_context = {
        "python_fundamentals": "Python language features, syntax, and best practices",
        "data_structures": "data structures, their implementations, and complexity analysis",
        "algorithms": "algorithmic problem-solving, optimization, and complexity",
        "machine_learning": "ML concepts, models, and practical implementation",
        "system_design": "system architecture, scalability, and distributed systems",
        "llm_engineering": "LLM APIs, prompt engineering, and AI application development"
    }
    
    prompt = f"""Generate {n_items} technical interview questions and answers.

Category: {category.replace('_', ' ').title()}
Context: {category_context.get(category, category)}
Difficulty: {difficulty} - {difficulty_guidance.get(difficulty, '')}

For each item, provide:
1. A clear, specific interview question
2. A comprehensive answer (2-4 paragraphs)
3. A code example if applicable (Python)
4. Two follow-up questions

Return as JSON with this structure:
{{
  "items": [
    {{
      "question": "...",
      "answer": "...",
      "code_example": "...",
      "follow_up_questions": ["...", "..."]
    }}
  ]
}}

Requirements:
- Questions should be realistic interview questions
- Answers should be thorough but concise
- Code examples should be practical and correct
- Return ONLY valid JSON, no markdown or explanations"""
    
    return prompt


def extract_json(text: str) -> Dict:
    """Extract JSON from LLM response."""
    # Remove markdown code blocks if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    
    # Try direct parsing
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Try to find JSON object
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1:
        try:
            return json.loads(text[start:end+1])
        except json.JSONDecodeError:
            pass
    
    raise ValueError(f"Could not parse JSON from response: {text[:200]}...")


def generate_llm_batch(category: str, difficulty: str, n_items: int = 5) -> List[Dict]:
    """Generate a batch of Q&A items using LLM."""
    if not USE_LLM:
        print("LLM not available, using rule-based fallback")
        return []
    
    prompt = create_generation_prompt(category, difficulty, n_items)
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a technical interview expert. Generate realistic interview Q&A in JSON format."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=4000
        )
        
        content = response.choices[0].message.content
        data = extract_json(content)
        
        items = data.get("items", [])
        
        # Add metadata to each item
        for item in items:
            item["id"] = str(uuid.uuid4())
            item["category"] = category
            item["difficulty"] = difficulty
            item["tags"] = generate_tags(category, difficulty)
            item["estimated_time_minutes"] = estimate_time(difficulty, category)
        
        return items
        
    except Exception as e:
        print(f"LLM generation error: {e}")
        return []


print("LLM generator functions defined")

In [ ]:
# Test LLM Generation

if USE_LLM:
    print("Testing LLM generation...")
    test_items = generate_llm_batch("python_fundamentals", "medium", n_items=2)
    
    if test_items:
        print(f"Generated {len(test_items)} items")
        for i, item in enumerate(test_items, 1):
            print(f"\n--- Item {i} ---")
            print(f"Q: {item['question'][:100]}...")
            print(f"A: {item['answer'][:150]}...")
    else:
        print("No items generated")
else:
    print("Skipping LLM test - no API key available")

In [ ]:
# Full Dataset Generator

def generate_full_dataset(config: Dict, use_llm: bool = True) -> pd.DataFrame:
    """Generate complete dataset with progress tracking."""
    
    total_rows = config["total_rows"]
    batch_size = config["batch_size"]
    all_items = []
    
    print(f"Generating {total_rows} items...")
    
    # Calculate items per category based on weights
    for cat_info in config["categories"]:
        category = cat_info["name"]
        cat_count = int(total_rows * cat_info["weight"])
        
        print(f"\nCategory: {category} ({cat_count} items)")
        
        for diff_info in config["difficulty_levels"]:
            difficulty = diff_info["level"]
            diff_count = int(cat_count * diff_info["weight"])
            
            if diff_count == 0:
                continue
            
            print(f"  {difficulty}: {diff_count} items", end=" ")
            
            if use_llm and USE_LLM:
                # Generate in batches
                remaining = diff_count
                while remaining > 0:
                    batch = min(batch_size, remaining)
                    items = generate_llm_batch(category, difficulty, batch)
                    
                    if items:
                        all_items.extend(items)
                        remaining -= len(items)
                        print(".", end="", flush=True)
                    else:
                        # Fallback to rule-based
                        for _ in range(remaining):
                            row = generate_rule_based_row(config)
                            row["category"] = category
                            row["difficulty"] = difficulty
                            all_items.append(row)
                        remaining = 0
                    
                    time.sleep(0.5)  # Rate limiting
            else:
                # Pure rule-based
                for _ in range(diff_count):
                    row = generate_rule_based_row(config)
                    row["category"] = category
                    row["difficulty"] = difficulty
                    all_items.append(row)
                print("(rule-based)", end="")
            
            print()
    
    df = pd.DataFrame(all_items)
    print(f"\nTotal generated: {len(df)} items")
    
    return df


print("Full generator defined")

In [ ]:
# Generate the Dataset

# For testing, use smaller config
TEST_CONFIG = CONFIG.copy()
TEST_CONFIG["total_rows"] = 20  # Smaller for testing
TEST_CONFIG["batch_size"] = 5

print("Generating test dataset...")
df = generate_full_dataset(TEST_CONFIG, use_llm=USE_LLM)

print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nCategory distribution:")
print(df["category"].value_counts())
print(f"\nDifficulty distribution:")
print(df["difficulty"].value_counts())

In [ ]:
# Preview Generated Data

print("Sample questions:\n")
for idx, row in df.head(5).iterrows():
    print(f"[{row['category']} - {row['difficulty']}]")
    print(f"Q: {row['question']}")
    if row['answer'] and not row['answer'].startswith('[Placeholder'):
        print(f"A: {row['answer'][:200]}...")
    print(f"Tags: {row['tags']}")
    print(f"Time: {row['estimated_time_minutes']} mins")
    print("-" * 50)

In [ ]:
# Data Validation

def validate_dataset(df: pd.DataFrame) -> Dict[str, Any]:
    """Validate the generated dataset."""
    
    report = {
        "total_rows": len(df),
        "valid_rows": 0,
        "issues": []
    }
    
    # Check required fields
    required = ["id", "category", "difficulty", "question", "answer"]
    for field in required:
        if field not in df.columns:
            report["issues"].append(f"Missing required field: {field}")
    
    # Check for null values in required fields
    for field in required:
        if field in df.columns:
            null_count = df[field].isna().sum()
            if null_count > 0:
                report["issues"].append(f"{field}: {null_count} null values")
    
    # Check category values
    valid_categories = [c["name"] for c in CONFIG["categories"]]
    invalid_cats = df[~df["category"].isin(valid_categories)]["category"].unique()
    if len(invalid_cats) > 0:
        report["issues"].append(f"Invalid categories: {invalid_cats}")
    
    # Check difficulty values
    valid_difficulties = [d["level"] for d in CONFIG["difficulty_levels"]]
    invalid_diffs = df[~df["difficulty"].isin(valid_difficulties)]["difficulty"].unique()
    if len(invalid_diffs) > 0:
        report["issues"].append(f"Invalid difficulties: {invalid_diffs}")
    
    # Check question/answer lengths
    short_questions = (df["question"].str.len() < 20).sum()
    if short_questions > 0:
        report["issues"].append(f"{short_questions} questions are too short")
    
    # Calculate valid rows
    report["valid_rows"] = len(df) - len(report["issues"])
    report["validation_passed"] = len(report["issues"]) == 0
    
    return report


# Run validation
validation_report = validate_dataset(df)
print("Validation Report:")
print(f"  Total rows: {validation_report['total_rows']}")
print(f"  Validation passed: {validation_report['validation_passed']}")
if validation_report["issues"]:
    print("  Issues:")
    for issue in validation_report["issues"]:
        print(f"    - {issue}")

In [ ]:
# Save Dataset

# Create output directory
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

# Save as CSV
csv_path = output_dir / f"technical_interview_qa_{timestamp}.csv"
df.to_csv(csv_path, index=False)
print(f"Saved CSV: {csv_path}")

# Save as JSON (preserves lists)
json_path = output_dir / f"technical_interview_qa_{timestamp}.json"
df.to_json(json_path, orient="records", indent=2)
print(f"Saved JSON: {json_path}")

# Save config for reproducibility
config_path = output_dir / f"config_{timestamp}.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Saved config: {config_path}")

print(f"\nDataset generation complete!")

In [ ]:
# Summary Statistics

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"\nTotal items: {len(df)}")
print(f"\nBy Category:")
for cat, count in df["category"].value_counts().items():
    print(f"  {cat}: {count} ({count/len(df)*100:.1f}%)")

print(f"\nBy Difficulty:")
for diff, count in df["difficulty"].value_counts().items():
    print(f"  {diff}: {count} ({count/len(df)*100:.1f}%)")

print(f"\nAverage estimated time: {df['estimated_time_minutes'].mean():.1f} minutes")

print(f"\nFiles saved:")
print(f"  - {csv_path}")
print(f"  - {json_path}")
print(f"  - {config_path}")